# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Freestyle direction: Growth / Recovery / Momentum Prediction


i want a challenge and to find if i am ready for real work or not and it can help the client on deciding how to use his resources efficiently

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Predicting page trends can help the content owner make decisions. For example, if a page is falling, it tells them that they might need to make changes to help improve its trend.

However, if a prediction is made incorrectly, it can cause a rising page to fall. Furthermore, failing to find a falling page will make the owner lose opportunities and waste resources and time on the wrong page

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [49]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv(r"D:\Programing\flyrank-ml-internship-assignments\data\raw\content_refresh_anonymized.csv")

In [50]:
df.head()


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [51]:
visible_pages = df[df['impressions_90d'] >= 100]

print(f"Original rows: {len(df)}")
print(f"Trustworthy rows: {len(visible_pages)}")

Original rows: 30000
Trustworthy rows: 22006


In [52]:
valid_trends = ['down', 'stable', 'up']
filtered_data = visible_pages[visible_pages['trend_direction'].isin(valid_trends)]
filtered_data.info()

<class 'pandas.DataFrame'>
Index: 21758 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              21758 non-null  str    
 1   client_id               21758 non-null  str    
 2   search_volume           21201 non-null  float64
 3   competition             21201 non-null  float64
 4   competition_level       21087 non-null  str    
 5   cpc                     21201 non-null  float64
 6   content_type            21758 non-null  str    
 7   main_intent             21272 non-null  str    
 8   word_count              15227 non-null  float64
 9   char_count              15227 non-null  float64
 10  provider_used           6143 non-null   str    
 11  model_used              16448 non-null  str    
 12  impressions_90d         21758 non-null  int64  
 13  clicks_90d              21758 non-null  int64  
 14  pageviews_90d           21758 non-null  int64  
 15  s

In [53]:
filtered_data['trend_direction'].value_counts(normalize=True)

trend_direction
down      0.604467
stable    0.240463
up        0.155069
Name: proportion, dtype: float64

In [54]:
age_proof = pd.crosstab(
    index=filtered_data['age_tier'], 
    columns=filtered_data['trend_direction'], 
    normalize='index' 
)

print(age_proof.mul(100).round(1))

trend_direction  down  stable    up
age_tier                           
181-365          60.8    25.1  14.1
31-90            68.3    14.2  17.5
365+             42.9    34.5  22.6
91-180           70.8    16.9  12.3


In [55]:
print(filtered_data.groupby(['trend_direction'])['search_volume'].agg(['mean','median']))


                       mean  median
trend_direction                    
down             130.729370    10.0
stable           209.511765    10.0
up               236.101591    10.0


In [56]:
print(filtered_data.groupby('trend_direction')['search_volume'].quantile([0.75, 0.9, 0.95]))

# Option B: how many pages are "high volume" per trend
high_vol = filtered_data['search_volume'] > filtered_data['search_volume'].quantile(0.9)
print(pd.crosstab(filtered_data['trend_direction'], high_vol, normalize='index'))

trend_direction      
down             0.75     20.0
                 0.90     90.0
                 0.95    260.0
stable           0.75     30.0
                 0.90    170.0
                 0.95    480.0
up               0.75     40.0
                 0.90    210.0
                 0.95    590.0
Name: search_volume, dtype: float64
search_volume       False     True 
trend_direction                    
down             0.919860  0.080140
stable           0.886277  0.113723
up               0.871962  0.128038


In [57]:
print(filtered_data.groupby('trend_direction')['days_since_last_update'].quantile([0.75, 0.9, 0.95]))

low_refresh = filtered_data['days_since_last_update'] < filtered_data['days_since_last_update'].quantile(0.1)
print(pd.crosstab(filtered_data['trend_direction'], low_refresh, normalize='index'))

trend_direction      
down             0.75    104.0
                 0.90    104.0
                 0.95    104.0
stable           0.75    104.0
                 0.90    104.0
                 0.95    104.0
up               0.75    104.0
                 0.90    104.0
                 0.95    104.0
Name: days_since_last_update, dtype: float64
days_since_last_update     False     True 
trend_direction                           
down                    0.916743  0.083257
stable                  0.910168  0.089832
up                      0.913159  0.086841


In [58]:
position_proof = pd.crosstab(
    index=filtered_data['position_tier'],
    columns=filtered_data['trend_direction'],
    normalize='index'
)
print(position_proof.mul(100).round(1))

trend_direction  down  stable    up
position_tier                      
deep             32.7    17.8  49.4
page_1           61.2    25.9  12.9
page_3_5         59.0    23.1  17.9
striking         63.5    23.8  12.8
top_3            76.3    18.0   5.7


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

Most pages in the filtered dataset trend down (60.4%), against 24.0% stable and 15.5% up. A few patterns stand out:

- **Age:** Content older than a year (365+ days) shows a notably lower down-rate and higher up-rate than any younger tier — but there's no smooth gradient among the younger tiers, so this looks like an "old content is more resilient" effect specifically, not a general aging trend.
- **Search volume:** The typical page shows no relationship between search volume and trend (medians are identical). The link only appears among high-volume outliers — up-trending pages are overrepresented in the top 10% of search volume (12.8% vs. 8.0% for down-trending pages).
- **Position tier:** Deep pages are far more likely to be trending up than top-ranked pages (49.4% vs. 5.7%), but this is likely driven by a ceiling/floor effect — top pages have little room left to rise, deep pages have little room left to fall — rather than deep pages being inherently more successful.
- **Update recency:** Checked as a candidate signal, but showed no meaningful difference across trend groups once outliers were accounted for — ruled out.

**What this can and can't say:** these are observed associations in this dataset, not causal proof — e.g., we can't say aging *causes* resilience or that low search volume *causes* decline, only that they co-occur. This also isn't predicting actual Google rankings, only modeling patterns in this client's data. Any model built from this would be decision-support (flagging pages worth a human look), not an automated verdict.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.